# 02 — Arrival Feature Engineering (V9.0)

**Pre-split特征计算：Network → Gate/Terminal → FAA → Body Type → Weather → V9.0新特征 → Save**

Post-split特征（Lag、Aircraft Continuity、Target Encoding）在 NB03 中计算。

In [1]:
# Standard imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# === Project paths (RCF: only change BASE_DIR) ===
BASE_DIR = Path('../../..')             # RCF: Path('/path/to/project')
sys.path.insert(0, str(BASE_DIR / 'src'))

# Import feature engineering modules
from features.network_features import add_network_features, add_route_risk_score
from features.weather_features import add_weather_severity_score
from features.origin_lookup import ORIGIN_STATS, HIGH_RISK_ORIGINS
from sklearn.preprocessing import LabelEncoder

# Paths
DATA_RAW = BASE_DIR / 'data' / 'raw' / 'LGA_Dataset'
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'

print("Feature engineering modules loaded.")

Feature engineering modules loaded.


## 1. Load Processed Data

In [2]:
# Load processed data from NB01
df = pd.read_parquet(DATA_PROCESSED / 'arrivals_processed.parquet')
print(f"Loaded arrivals: {len(df):,} records")

# Load auxiliary data
df_departures = None
dep_path = DATA_PROCESSED / 'departures_processed.parquet'
if dep_path.exists():
    df_departures = pd.read_parquet(dep_path)
    print(f"Loaded departures: {len(df_departures):,} records")

df_missed = None
ma_path = DATA_PROCESSED / 'missed_approaches_processed.parquet'
if ma_path.exists():
    df_missed = pd.read_parquet(ma_path)
    print(f"Loaded missed approaches: {len(df_missed):,} records")

# Load FAA data
faa_data = {}
for name in ['gdp', 'gsp', 'dd', 'ad']:
    faa_path = DATA_PROCESSED / f'faa_{name}_processed.parquet'
    if faa_path.exists():
        faa_data[name] = pd.read_parquet(faa_path)
        print(f"Loaded FAA {name.upper()}: {len(faa_data[name]):,} records")
    else:
        faa_data[name] = None
        print(f"FAA {name.upper()}: not available")

print(f"\nDate range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Total arrivals: {len(df):,}")

Loaded arrivals: 148,165 records
Loaded departures: 148,061 records
Loaded missed approaches: 1,051 records
Loaded FAA GDP: 67 records
Loaded FAA GSP: 271 records
Loaded FAA DD: 692 records
Loaded FAA AD: 55 records

Date range: 2024-12-31 to NaT
Total arrivals: 148,165


## 2. Network Features

Features from Network Analysis:
- `origin_impact_score`: Total delay impact score (volume × avg delay)
- `origin_historical_delay`: Historical average arrival delay from this origin
- `origin_severe_delay_rate`: Percentage of flights with severe delays (>6h)
- `is_high_risk_origin`: Binary flag for high-risk origin airports

In [3]:
# Add network features
df = add_network_features(df, origin_col='Non-PA Airport', verbose=True)

# Add route risk score
df = add_route_risk_score(df, origin_col='Non-PA Airport')

# NOTE: add_origin_tier_feature removed - origin_tier had low SHAP importance

Network Features Added:
  - origin_impact_score: mean=20238.71, max=68130.00
  - origin_historical_delay: mean=16.16 min
  - is_high_risk_origin: 59538 flights (40.2%)


In [4]:
# Validate network features (is_high_risk_origin kept for route_risk_score, not as standalone feature)
network_features = ['origin_impact_score', 'origin_historical_delay', 
                    'origin_risk_level', 'route_risk_score']

print("Network Feature Statistics:")
print(df[network_features].describe())

Network Feature Statistics:
       origin_impact_score  origin_historical_delay  origin_risk_level  \
count        148165.000000            148165.000000      148165.000000   
mean          20238.712759                16.159630           0.483468   
std           17553.933406                11.284076           0.845144   
min            -600.000000                -6.382979           0.000000   
25%            6820.000000                11.289377           0.000000   
50%           14961.000000                15.373824           0.000000   
75%           32772.000000                20.495310           1.000000   
max           68130.000000               179.285714           4.000000   

       route_risk_score  
count     148165.000000  
mean           0.170095  
std            0.140539  
min           -0.020320  
25%            0.054566  
50%            0.081702  
75%            0.324117  
max            0.508489  


## 3. Gate & Terminal Encoding

Encode Gate and Terminal Code as numeric features for the model.
SHAP analysis will determine if these are useful.

**NOTE:** Lag features (`delay_rolling_1h`, `delay_rolling_3h`, etc.) and congestion features
are now computed in `03_model_training.ipynb` AFTER train/test split to prevent data leakage.

In [5]:
# Gate: label encode raw Gate column (SHAP will judge if useful)
le_gate = LabelEncoder()
df['gate_encoded'] = le_gate.fit_transform(df['Gate'].fillna('Unknown'))

# Terminal Code: label encode for model use
le_terminal = LabelEncoder()
df['terminal_encoded'] = le_terminal.fit_transform(
    df['Terminal Code'].fillna('Unknown')
)

print(f"Gate encoding: {df['gate_encoded'].nunique()} unique values")
print(f"Terminal encoding: {df['terminal_encoded'].nunique()} unique values")
print(f"\nTerminal Code distribution:")
print(df['Terminal Code'].value_counts())

Gate encoding: 110 unique values
Terminal encoding: 5 unique values

Terminal Code distribution:
Terminal Code
Terminal B     70394
Terminal C     69873
Terminal A      4827
LGA-Unknown     2448
Unknown          623
Name: count, dtype: int64


## 4. FAA Operational Features

Match FAA Traffic Management events to flights using Scheduled Arrival Time.
- **GDP**: Flight falls within GDP Start → Effective End window → `is_gdp_active`, `gdp_avg_delay`
- **GSP**: Flight falls within GSP Start → Effective End window → `is_ground_stop`
- **DD**: Flight within ±1h of DD Update Time → `is_dep_delay_active`
- **Composite**: `faa_delay_severity` (0=None, 1=DD, 2=GDP, 3=GS), `faa_delay_reason_raw`

**No leakage**: FAA advisories are public data issued BEFORE flights arrive. We match on scheduled time.

In [6]:
# FAA Event Matching — match flights to active FAA programs
# Uses Scheduled_Arrival_Datetime for matching (no leakage)

arr_dt = pd.to_datetime(df['Scheduled_Arrival_Datetime'])

# Initialize FAA columns
df['is_gdp_active'] = 0
df['gdp_avg_delay'] = 0.0
df['is_ground_stop'] = 0
df['is_dep_delay_active'] = 0

# --- GDP matching (window: GDP_Start → GDP_Effective_End) ---
df_gdp = faa_data.get('gdp')
if df_gdp is not None and len(df_gdp) > 0:
    gdp_matched = 0
    for _, row in df_gdp.iterrows():
        start = row.get('GDP_Start')
        end = row.get('GDP_Effective_End')
        if pd.isna(start) or pd.isna(end):
            continue
        mask = (arr_dt >= start) & (arr_dt <= end)
        n_match = mask.sum()
        if n_match > 0:
            df.loc[mask, 'is_gdp_active'] = 1
            # Keep max delay if multiple GDPs overlap
            avg_delay = row.get('GDP_Avg_Delay_Min', 0)
            if pd.notna(avg_delay):
                df.loc[mask, 'gdp_avg_delay'] = df.loc[mask, 'gdp_avg_delay'].clip(lower=avg_delay)
            gdp_matched += n_match
    print(f"GDP: {len(df_gdp)} events → {df['is_gdp_active'].sum():,} flights matched "
          f"({df['is_gdp_active'].mean()*100:.1f}%)")
else:
    print("GDP: no data available")

# --- GSP matching (window: GSP_Start → GSP_Effective_End) ---
df_gsp = faa_data.get('gsp')
if df_gsp is not None and len(df_gsp) > 0:
    for _, row in df_gsp.iterrows():
        start = row.get('GSP_Start')
        end = row.get('GSP_Effective_End')
        if pd.isna(start) or pd.isna(end):
            continue
        mask = (arr_dt >= start) & (arr_dt <= end)
        df.loc[mask, 'is_ground_stop'] = 1
    print(f"GSP: {len(df_gsp)} events → {df['is_ground_stop'].sum():,} flights matched "
          f"({df['is_ground_stop'].mean()*100:.1f}%)")
else:
    print("GSP: no data available")

# --- DD matching (±1h window from DD_Update) ---
df_dd = faa_data.get('dd')
if df_dd is not None and len(df_dd) > 0:
    window = pd.Timedelta(hours=1)
    for _, row in df_dd.iterrows():
        update_time = row.get('DD_Update')
        if pd.isna(update_time):
            continue
        mask = (arr_dt >= update_time - window) & (arr_dt <= update_time + window)
        df.loc[mask, 'is_dep_delay_active'] = 1
    print(f"DD: {len(df_dd)} events → {df['is_dep_delay_active'].sum():,} flights matched "
          f"({df['is_dep_delay_active'].mean()*100:.1f}%)")
else:
    print("DD: no data available")

# --- Composite: faa_delay_severity (0=None, 1=DD only, 2=GDP, 3=Ground Stop) ---
# Higher severity takes priority
df['faa_delay_severity'] = 0
df.loc[df['is_dep_delay_active'] == 1, 'faa_delay_severity'] = 1
df.loc[df['is_gdp_active'] == 1, 'faa_delay_severity'] = 2
df.loc[df['is_ground_stop'] == 1, 'faa_delay_severity'] = 3

# --- faa_delay_reason_raw: extract primary reason category ---
# Build reason from GDP/GSP Reason columns (for target encoding in 03)
df['faa_delay_reason_raw'] = 'None'

# Map GDP reasons
if df_gdp is not None and 'GDP Reason' in df_gdp.columns:
    for _, row in df_gdp.iterrows():
        start = row.get('GDP_Start')
        end = row.get('GDP_Effective_End')
        reason = str(row.get('GDP Reason', '')).strip()
        if pd.isna(start) or pd.isna(end) or not reason:
            continue
        mask = (arr_dt >= start) & (arr_dt <= end)
        # Categorize reason
        reason_lower = reason.lower()
        if any(w in reason_lower for w in ['thunder', 'storm', 'wind', 'weather', 'ceiling', 'low ceil']):
            cat = 'WX'
        elif any(w in reason_lower for w in ['volume', 'vol']):
            cat = 'VOL'
        elif any(w in reason_lower for w in ['runway', 'rwy']):
            cat = 'RWY'
        elif any(w in reason_lower for w in ['staff']):
            cat = 'STAFF'
        else:
            cat = 'OTHER'
        # Only overwrite 'None' entries (don't downgrade GSP reason)
        update_mask = mask & (df['faa_delay_reason_raw'] == 'None')
        df.loc[update_mask, 'faa_delay_reason_raw'] = cat

# Map GSP reasons (higher priority than GDP)
if df_gsp is not None and 'GSP Reason' in df_gsp.columns:
    for _, row in df_gsp.iterrows():
        start = row.get('GSP_Start')
        end = row.get('GSP_Effective_End')
        reason = str(row.get('GSP Reason', '')).strip()
        if pd.isna(start) or pd.isna(end) or not reason:
            continue
        mask = (arr_dt >= start) & (arr_dt <= end)
        reason_lower = reason.lower()
        if any(w in reason_lower for w in ['thunder', 'storm', 'wind', 'weather', 'ceiling', 'low ceil']):
            cat = 'WX'
        elif any(w in reason_lower for w in ['volume', 'vol']):
            cat = 'VOL'
        elif any(w in reason_lower for w in ['runway', 'rwy']):
            cat = 'RWY'
        elif any(w in reason_lower for w in ['staff']):
            cat = 'STAFF'
        else:
            cat = 'OTHER'
        df.loc[mask, 'faa_delay_reason_raw'] = cat  # GSP overwrites GDP

print(f"\nFAA delay severity distribution:")
print(df['faa_delay_severity'].value_counts().sort_index())
print(f"\nFAA delay reason distribution:")
print(df['faa_delay_reason_raw'].value_counts())

GDP: 67 events → 10,719 flights matched (7.2%)
GSP: 271 events → 6,239 flights matched (4.2%)
DD: 692 events → 22,390 flights matched (15.1%)

FAA delay severity distribution:
faa_delay_severity
0    118398
1     14732
2      8796
3      6239
Name: count, dtype: int64

FAA delay reason distribution:
faa_delay_reason_raw
None     133130
WX        12567
OTHER       897
STAFF       817
VOL         728
RWY          26
Name: count, dtype: int64


## 5. Body Type Encoding

One-hot encode aircraft Body Type (Narrow Body, Commuter, Wide Body). Wide Body is the reference category.

In [7]:
# Body Type one-hot encoding (Wide Body = reference category, both columns = 0)
if 'Body Type Desc' in df.columns:
    df['body_type_narrow'] = (df['Body Type Desc'] == 'Narrow Body').astype(int)
    df['body_type_commuter'] = (df['Body Type Desc'] == 'Commuter').astype(int)

    print("Body Type distribution:")
    print(df['Body Type Desc'].value_counts())
    print(f"\nbody_type_narrow: {df['body_type_narrow'].sum():,} ({df['body_type_narrow'].mean()*100:.1f}%)")
    print(f"body_type_commuter: {df['body_type_commuter'].sum():,} ({df['body_type_commuter'].mean()*100:.1f}%)")
    print(f"Wide Body (reference): {((df['body_type_narrow']==0) & (df['body_type_commuter']==0)).sum():,}")
else:
    df['body_type_narrow'] = 0
    df['body_type_commuter'] = 0
    print("WARNING: 'Body Type Desc' column not found — body type features set to 0")

Body Type distribution:
Body Type Desc
Commuter            26470
Narrow Body         22368
Unknown              1067
General Aviation      584
Name: count, dtype: int64

body_type_narrow: 22,368 (15.1%)
body_type_commuter: 26,470 (17.9%)
Wide Body (reference): 99,327


In [8]:
# Load LGA weather data — V9.0: 3 sources (1 old aggregated + 2 new WeatherFull)
# Old: LGA_Summer2025_Weather.csv (May-Sept, pre-aggregated hourly)
# New: LGA_Jan-May2025_WeatherFull.csv, LGA_Sept-Oct2025_WeatherFull.csv (raw observations)

weather_file_old = DATA_RAW / 'LGA_Summer2025_Weather.csv'
weather_files_new = [
    DATA_RAW / 'LGA_Jan-May2025_WeatherFull.csv',
    DATA_RAW / 'LGA_Sept-Oct2025_WeatherFull.csv',
]

# Load old format
df_weather_old = None
if weather_file_old.exists():
    df_weather_old = pd.read_csv(weather_file_old)
    print(f"LGA Weather (old format): {len(df_weather_old)} records")

# Load new WeatherFull format
weather_full_dfs = []
for f in weather_files_new:
    if f.exists():
        wf = pd.read_csv(f)
        # Filter to LGA only (WeatherFull contains only LGA but key column confirms)
        if 'key' in wf.columns:
            wf = wf[wf['key'] == 'KLGA']
        weather_full_dfs.append(wf)
        print(f"LGA Weather (new format): {f.name} — {len(wf)} records")
    else:
        print(f"WARNING: {f.name} not found — skipping")

if weather_full_dfs:
    df_weather_full = pd.concat(weather_full_dfs, ignore_index=True)
    print(f"LGA WeatherFull total: {len(df_weather_full)} records")
else:
    df_weather_full = None
    print("No WeatherFull files found")

LGA Weather (old format): 2800 records
LGA Weather (new format): LGA_Jan-May2025_WeatherFull.csv — 3963 records
LGA Weather (new format): LGA_Sept-Oct2025_WeatherFull.csv — 1623 records
LGA WeatherFull total: 5586 records


In [9]:
# Load Origin Weather Data — V9.0: 4 files (all now locally available)
origin_weather_files = [
    DATA_RAW / 'LGA_NonPAAirportWeather_Jan-May2025.csv',      # 265MB, NEW
    DATA_RAW / 'LGA_NonPAAirportWeather_May-June2025.csv',     # 71MB
    DATA_RAW / 'LGA_NonPAAirportWeather_July-Sept2025.csv',    # 125MB
    DATA_RAW / 'LGA_NonPAAirportWeather_Sept-Oct2025.csv',     # 129MB, NEW
]

# Load each file individually to manage memory (~2.95M rows total)
origin_dfs = []
for f in origin_weather_files:
    if f.exists():
        chunk = pd.read_csv(f)
        origin_dfs.append(chunk)
        print(f"Loaded: {f.name} — {len(chunk):,} rows")
    else:
        print(f"NOT FOUND: {f.name}")

if origin_dfs:
    df_origin_weather = pd.concat(origin_dfs, ignore_index=True)
    print(f"\nTotal origin weather records: {len(df_origin_weather):,}")
    print(f"Unique airports: {df_origin_weather['obs_id'].nunique()}")
else:
    df_origin_weather = None
    print("Origin weather files not found - skipping")

Loaded: LGA_NonPAAirportWeather_Jan-May2025.csv — 1,324,881 rows
Loaded: LGA_NonPAAirportWeather_May-June2025.csv — 354,979 rows
Loaded: LGA_NonPAAirportWeather_July-Sept2025.csv — 629,331 rows
Loaded: LGA_NonPAAirportWeather_Sept-Oct2025.csv — 646,520 rows

Total origin weather records: 2,955,711
Unique airports: 606


## 6. Weather Data

### 6.1 LGA Weather

In [10]:
def prepare_weather_features(df_weather):
    """Process LGA weather data (OLD aggregated format), aggregate by hour."""
    weather = df_weather.copy()
    weather['Date_part'] = pd.to_datetime(weather['Date']).dt.date
    mil_time = pd.to_datetime(weather['Military Time']).dt.time
    weather['Datetime'] = pd.to_datetime(
        weather['Date_part'].astype(str) + ' ' + mil_time.astype(str)
    )
    weather['Timestamp'] = weather['Datetime'].dt.floor('h')
    
    STORM_KEYWORDS = ['Thunder', 'Storm', 'T-Storm']
    RAIN_KEYWORDS = ['Rain', 'Drizzle', 'Shower']
    FOG_KEYWORDS = ['Fog', 'Haze', 'Mist']
    
    def check_keywords(desc, keywords):
        if pd.isna(desc):
            return 0
        return int(any(k.lower() in str(desc).lower() for k in keywords))
    
    weather['_storm'] = weather['Weather Desc'].apply(lambda x: check_keywords(x, STORM_KEYWORDS))
    weather['_rain'] = weather['Weather Desc'].apply(lambda x: check_keywords(x, RAIN_KEYWORDS))
    weather['_fog'] = weather['Weather Desc'].apply(lambda x: check_keywords(x, FOG_KEYWORDS))
    
    weather_hourly = weather.groupby('Timestamp').agg({
        'Total Precipitation (In)': 'sum',
        'Avg Wind Gusts (mph)': 'max',
        'Avg Visibility (Miles)': 'min',
        '_storm': 'max', '_rain': 'max', '_fog': 'max',
    }).reset_index()
    
    weather_hourly.columns = ['Timestamp', 'Precipitation', 'Wind_Gusts', 'Visibility',
                               'lga_storm_flag', 'lga_rain_flag', 'lga_fog_flag']
    weather_hourly['Is_Bad_Weather'] = (
        (weather_hourly['Visibility'] < 5) | 
        (weather_hourly['Precipitation'] > 0.1) |
        (weather_hourly['lga_storm_flag'] == 1)
    ).astype(int)
    return weather_hourly


def prepare_lga_weather_from_full(df_weather_full):
    """Process LGA WeatherFull format (same structure as NonPAAirportWeather).
    Aggregates raw observations to hourly, matching canonical LGA weather columns."""
    weather = df_weather_full.copy()
    
    # Parse time — use valid_time_est (local time), strip timezone to match old format
    dt = pd.to_datetime(weather['valid_time_est'], errors='coerce')
    if dt.dt.tz is not None:
        dt = dt.dt.tz_localize(None)
    weather['Datetime'] = dt
    weather['Timestamp'] = weather['Datetime'].dt.floor('h')
    
    # Extract weather flags from wx_phrase
    STORM_KEYWORDS = ['Thunder', 'Storm', 'T-Storm']
    RAIN_KEYWORDS = ['Rain', 'Drizzle', 'Shower']
    FOG_KEYWORDS = ['Fog', 'Haze', 'Mist']
    
    weather['_storm'] = weather['wx_phrase'].str.contains(
        '|'.join(STORM_KEYWORDS), case=False, na=False).astype(int)
    weather['_rain'] = weather['wx_phrase'].str.contains(
        '|'.join(RAIN_KEYWORDS), case=False, na=False).astype(int)
    weather['_fog'] = weather['wx_phrase'].str.contains(
        '|'.join(FOG_KEYWORDS), case=False, na=False).astype(int)
    
    # Ensure numeric types
    for col in ['precip_hrly', 'gust', 'vis']:
        weather[col] = pd.to_numeric(weather[col], errors='coerce')
    
    # Aggregate by hour
    weather_hourly = weather.groupby('Timestamp').agg({
        'precip_hrly': 'sum',
        'gust': 'max',
        'vis': 'min',
        '_storm': 'max', '_rain': 'max', '_fog': 'max',
    }).reset_index()
    
    # Map to canonical LGA weather column names
    weather_hourly.columns = ['Timestamp', 'Precipitation', 'Wind_Gusts', 'Visibility',
                               'lga_storm_flag', 'lga_rain_flag', 'lga_fog_flag']
    weather_hourly['Is_Bad_Weather'] = (
        (weather_hourly['Visibility'] < 5) | 
        (weather_hourly['Precipitation'] > 0.1) |
        (weather_hourly['lga_storm_flag'] == 1)
    ).astype(int)
    return weather_hourly


# === Process & Merge LGA Weather ===
weather_parts = []

# Old format (May-Sept)
if df_weather_old is not None:
    w_old = prepare_weather_features(df_weather_old)
    weather_parts.append(w_old)
    print(f"Old format → {len(w_old)} hourly records ({w_old['Timestamp'].min()} to {w_old['Timestamp'].max()})")

# New WeatherFull format (Jan-May, Sept-Oct)
if df_weather_full is not None:
    w_new = prepare_lga_weather_from_full(df_weather_full)
    weather_parts.append(w_new)
    print(f"New format → {len(w_new)} hourly records ({w_new['Timestamp'].min()} to {w_new['Timestamp'].max()})")

# Combine and deduplicate
if weather_parts:
    weather_hourly = pd.concat(weather_parts, ignore_index=True)
    weather_hourly = weather_hourly.sort_values('Timestamp').drop_duplicates(
        subset='Timestamp', keep='first'
    ).reset_index(drop=True)
    print(f"\nUnified LGA weather: {len(weather_hourly)} hourly records")
    print(f"Date range: {weather_hourly['Timestamp'].min()} to {weather_hourly['Timestamp'].max()}")
else:
    weather_hourly = pd.DataFrame()
    print("WARNING: No LGA weather data available")

# Create merge key for flights
df['Timestamp'] = pd.to_datetime(df['Scheduled_Arrival_Datetime']).dt.floor('h')

# Merge weather features
if len(weather_hourly) > 0:
    df = df.merge(weather_hourly, on='Timestamp', how='left')

# Check coverage
weather_cols = ['Precipitation', 'Wind_Gusts', 'Visibility', 'Is_Bad_Weather',
                'lga_storm_flag', 'lga_rain_flag', 'lga_fog_flag']
print(f"\nWeather feature coverage:")
for col in weather_cols:
    if col in df.columns:
        coverage = df[col].notna().sum() / len(df) * 100
        print(f"  {col}: {coverage:.1f}% non-null")
        if col.endswith('_flag'):
            print(f"    positive: {df[col].sum():,} ({df[col].mean()*100:.1f}%)")

Old format → 2471 hourly records (2025-05-22 00:00:00 to 2025-09-01 23:00:00)
New format → 4825 hourly records (2025-01-01 00:00:00 to 2025-10-31 23:00:00)

Unified LGA weather: 7272 hourly records
Date range: 2025-01-01 00:00:00 to 2025-10-31 23:00:00

Weather feature coverage:
  Precipitation: 99.0% non-null
  Wind_Gusts: 99.0% non-null
  Visibility: 99.0% non-null
  Is_Bad_Weather: 99.0% non-null
  lga_storm_flag: 99.0% non-null
    positive: 587.0 (0.4%)
  lga_rain_flag: 99.0% non-null
    positive: 11,584.0 (7.9%)
  lga_fog_flag: 99.0% non-null
    positive: 3,516.0 (2.4%)


In [11]:
# Add weather severity score (per sponsor feedback 2026-02-04)
# Combines precipitation, wind, visibility into a single score
df = add_weather_severity_score(
    df,
    precip_col='Precipitation',
    wind_col='Wind_Gusts',
    visibility_col='Visibility',
    verbose=True
)

Weather Severity Features Added:
  - weather_severity_score: mean=0.087
  - weather_category distribution:
      GOOD: 143578 (96.9%)
      MODERATE: 3830 (2.6%)
      SEVERE: 757 (0.5%)


### 6.2 Origin Weather

In [12]:
def prepare_origin_weather(df_origin):
    """
    Process origin weather data, aggregate by airport + hour.
    
    Expanded columns (per plan v3, RCF-verified):
    - precip_hrly, gust, vis (existing)
    - pressure (0.3% null), dewPt, temp (new numeric)
    - pressure_desc → origin_pressure_rapid_flag (79.9% null, null=0)
    - wx_phrase → origin_storm_flag (3.9% positive)
    - clds → origin_cloud_cover (CLR=0..OVC=4)
    
    TIMEZONE NOTE: Data has valid_time_est (EST values with Z suffix) and valid_time_gmt (true UTC).
    Flight data is in LGA local time (EST/EDT), so we use valid_time_est for correct merge alignment.
    """
    weather = df_origin.copy()
    
    # Parse time — use valid_time_est to match flight data timezone (EST/EDT)
    # NOTE: valid_time_est has Z suffix but values are actually EST — strip tz to get naive EST
    dt = pd.to_datetime(weather['valid_time_est'], errors='coerce')
    if dt.dt.tz is not None:
        dt = dt.dt.tz_localize(None)
    weather['Datetime'] = dt
    weather['Timestamp'] = weather['Datetime'].dt.floor('h')
    weather['airport_code'] = weather['obs_id'].str.upper()
    
    # --- Numeric aggregation ---
    agg_cols = {
        'precip_hrly': 'sum',
        'gust': 'max',
        'vis': 'min',
    }
    # Add new columns if they exist
    for col in ['pressure', 'dewPt', 'temp']:
        if col in weather.columns:
            agg_cols[col] = 'mean'
    
    weather_hourly = weather.groupby(['airport_code', 'Timestamp']).agg(agg_cols).reset_index()
    
    # Rename columns
    rename_map = {
        'precip_hrly': 'origin_precip',
        'gust': 'origin_wind_gust',
        'vis': 'origin_visibility',
        'pressure': 'origin_pressure',
        'dewPt': 'origin_dewpoint',
        'temp': 'origin_temp',
    }
    weather_hourly.rename(columns=rename_map, inplace=True)
    
    # --- Pressure rapid change flag (from pressure_desc) ---
    # 79.9% null, but Rising/Falling Rapidly = thunderstorm signal per sponsor
    if 'pressure_desc' in weather.columns:
        weather['pressure_rapid'] = weather['pressure_desc'].isin(
            ['Rising Rapidly', 'Falling Rapidly']
        ).astype(int)
        rapid_flag = weather.groupby(['airport_code', 'Timestamp'])['pressure_rapid'].max().reset_index()
        weather_hourly = weather_hourly.merge(rapid_flag, on=['airport_code', 'Timestamp'], how='left')
        weather_hourly.rename(columns={'pressure_rapid': 'origin_pressure_rapid_flag'}, inplace=True)
        weather_hourly['origin_pressure_rapid_flag'] = weather_hourly['origin_pressure_rapid_flag'].fillna(0).astype(int)
    
    # --- Storm flag (from wx_phrase) ---
    # 3.9% of records contain Thunder/Storm (38,317 out of 984,310)
    if 'wx_phrase' in weather.columns:
        weather['has_storm'] = weather['wx_phrase'].str.contains(
            'Thunder|Storm|T-Storm', case=False, na=False
        ).astype(int)
        storm_flag = weather.groupby(['airport_code', 'Timestamp'])['has_storm'].max().reset_index()
        weather_hourly = weather_hourly.merge(storm_flag, on=['airport_code', 'Timestamp'], how='left')
        weather_hourly.rename(columns={'has_storm': 'origin_storm_flag'}, inplace=True)
        weather_hourly['origin_storm_flag'] = weather_hourly['origin_storm_flag'].fillna(0).astype(int)
    
    # --- Cloud cover encoding (from clds) ---
    # CLR=0, FEW=1, SCT=2, BKN=3, OVC=4
    if 'clds' in weather.columns:
        cloud_map = {'CLR': 0, 'FEW': 1, 'SCT': 2, 'BKN': 3, 'OVC': 4}
        weather['cloud_num'] = weather['clds'].map(cloud_map)
        cloud_agg = weather.groupby(['airport_code', 'Timestamp'])['cloud_num'].max().reset_index()
        weather_hourly = weather_hourly.merge(cloud_agg, on=['airport_code', 'Timestamp'], how='left')
        weather_hourly.rename(columns={'cloud_num': 'origin_cloud_cover'}, inplace=True)
    
    # --- Updated bad weather flag ---
    bad_conditions = (
        (weather_hourly['origin_visibility'] < 5) | 
        (weather_hourly['origin_precip'] > 0.1) |
        (weather_hourly['origin_wind_gust'] > 25)
    )
    if 'origin_pressure_rapid_flag' in weather_hourly.columns:
        bad_conditions = bad_conditions | (weather_hourly['origin_pressure_rapid_flag'] == 1)
    if 'origin_storm_flag' in weather_hourly.columns:
        bad_conditions = bad_conditions | (weather_hourly['origin_storm_flag'] == 1)
    weather_hourly['origin_bad_weather'] = bad_conditions.astype(int)
    
    return weather_hourly


if df_origin_weather is not None:
    # Process origin weather
    origin_hourly = prepare_origin_weather(df_origin_weather)
    print(f"Origin weather hourly: {len(origin_hourly):,} records")
    print(f"Airports covered: {origin_hourly['airport_code'].nunique()}")
    
    # Merge with flight data
    df['origin_airport_upper'] = df['Non-PA Airport'].str.upper()
    
    df = df.merge(
        origin_hourly,
        left_on=['origin_airport_upper', 'Timestamp'],
        right_on=['airport_code', 'Timestamp'],
        how='left'
    )
    
    # Clean up
    df.drop(columns=['origin_airport_upper', 'airport_code'], inplace=True, errors='ignore')
    
    # Check coverage
    origin_cols = [
        'origin_precip', 'origin_wind_gust', 'origin_visibility', 'origin_bad_weather',
        'origin_pressure', 'origin_dewpoint', 'origin_temp',
        'origin_pressure_rapid_flag', 'origin_storm_flag', 'origin_cloud_cover'
    ]
    print(f"\nOrigin weather feature coverage:")
    for col in origin_cols:
        if col in df.columns:
            coverage = df[col].notna().sum() / len(df) * 100
            print(f"  {col}: {coverage:.1f}% non-null")
            if col.endswith('_flag'):
                non_null = df[col].dropna()
                if len(non_null) > 0:
                    print(f"    positive: {int(non_null.sum()):,} ({non_null.mean()*100:.1f}%)")
else:
    print("Skipping origin weather features - data not available")

Origin weather hourly: 1,980,107 records
Airports covered: 606

Origin weather feature coverage:
  origin_precip: 31.0% non-null
  origin_wind_gust: 31.0% non-null
  origin_visibility: 31.0% non-null
  origin_bad_weather: 31.0% non-null
  origin_pressure: 31.0% non-null
  origin_dewpoint: 31.0% non-null
  origin_temp: 31.0% non-null
  origin_pressure_rapid_flag: 31.0% non-null
    positive: 7,914 (17.2%)
  origin_storm_flag: 31.0% non-null
    positive: 1,715 (3.7%)
  origin_cloud_cover: 31.0% non-null


### 6.3 Weather Impact Scores (V9.0)

Pre-computed weather impact scores (0-10 scale) from FAA/NOAA data.
- **origin_wx_impact**: Weather impact at origin airport (54 airports, 25.2% non-zero)
- **lga_wx_impact**: Weather impact at LGA (37.8% non-zero, winter/spring 44-47%)

Replaces low-signal features: `origin_storm_flag` (SHAP 0.0034), `origin_cloud_cover` (0.0203)

In [13]:
# === Load & Merge Weather Impact Scores (V9.0) ===
# Pre-computed wx_impact_score (0-10) for 54 airports, Jan-Oct 2025
wx_impact_path = DATA_RAW / 'WX_ImpactScores_2025_Full.csv'

if wx_impact_path.exists():
    df_wx_impact = pd.read_csv(wx_impact_path)
    df_wx_impact['Timestamp'] = (
        pd.to_datetime(df_wx_impact['date']) +
        pd.to_timedelta(df_wx_impact['hour'], unit='h')
    )
    print(f"Weather Impact Scores loaded: {len(df_wx_impact):,} records, "
          f"{df_wx_impact['airport'].nunique()} airports")
    print(f"Date range: {df_wx_impact['date'].min()} to {df_wx_impact['date'].max()}")
    print(f"Non-zero scores: {(df_wx_impact['wx_impact_score'] > 0).mean()*100:.1f}%")

    # --- origin_wx_impact: match on origin airport + hour ---
    origin_wx = df_wx_impact[['airport', 'Timestamp', 'wx_impact_score']].copy()
    origin_wx.rename(columns={'wx_impact_score': 'origin_wx_impact'}, inplace=True)

    df['_origin_upper'] = df['Non-PA Airport'].str.upper().str.strip()
    df = df.merge(
        origin_wx,
        left_on=['_origin_upper', 'Timestamp'],
        right_on=['airport', 'Timestamp'],
        how='left'
    )
    df['origin_wx_impact'] = df['origin_wx_impact'].fillna(0)
    df.drop(columns=['_origin_upper', 'airport'], inplace=True, errors='ignore')

    origin_match = (df['origin_wx_impact'] > 0).sum()
    print(f"\norigin_wx_impact: matched {origin_match:,} flights "
          f"({origin_match/len(df)*100:.1f}%), mean={df['origin_wx_impact'].mean():.2f}")

    # --- lga_wx_impact: match on LGA + hour ---
    lga_wx = df_wx_impact[df_wx_impact['airport'] == 'LGA'][['Timestamp', 'wx_impact_score']].copy()
    lga_wx.rename(columns={'wx_impact_score': 'lga_wx_impact'}, inplace=True)
    lga_wx = lga_wx.drop_duplicates(subset='Timestamp', keep='first')

    df = df.merge(lga_wx, on='Timestamp', how='left')
    df['lga_wx_impact'] = df['lga_wx_impact'].fillna(0)

    lga_match = (df['lga_wx_impact'] > 0).sum()
    print(f"lga_wx_impact: matched {lga_match:,} flights "
          f"({lga_match/len(df)*100:.1f}%), mean={df['lga_wx_impact'].mean():.2f}")
else:
    print(f"WARNING: {wx_impact_path} not found — wx_impact features set to 0")
    df['origin_wx_impact'] = 0
    df['lga_wx_impact'] = 0

Weather Impact Scores loaded: 381,024 records, 54 airports
Date range: 2025-01-01 to 2025-10-30
Non-zero scores: 25.2%

origin_wx_impact: matched 28,577 flights (19.3%), mean=0.72
lga_wx_impact: matched 56,050 flights (37.8%), mean=1.33


## 7. FAA Event Depth Features (V9.0)

In [14]:
# === V9.0: FAA Event Depth Features ===
# faa_event_duration_hours: max duration of active GDP/GSP at flight time
# faa_active_event_count: number of simultaneously active FAA events

# Parse GDP start/end times
faa_gdp = pd.read_csv(DATA_RAW / 'LGA_FAA_GrounDelays2025.csv')
faa_gsp = pd.read_csv(DATA_RAW / 'LGA_FAA_GroundStops2025.csv')

def parse_faa_times(faa_df, start_col, end_col):
    """Parse FAA event start/end times, return list of (start, end) tuples."""
    events = []
    for _, row in faa_df.iterrows():
        try:
            start = pd.to_datetime(row[start_col])
            end = pd.to_datetime(row[end_col])
            if pd.notna(start) and pd.notna(end) and end > start:
                events.append((start, end))
        except:
            continue
    return events

gdp_events = parse_faa_times(faa_gdp, 'GDP Start Time', 'GDP End Time')
gsp_events = parse_faa_times(faa_gsp, 'GSP Start Time', 'GSP End Time')
all_events = gdp_events + gsp_events

print(f"FAA events parsed: GDP={len(gdp_events)}, GSP={len(gsp_events)}, Total={len(all_events)}")

# For each flight, find active events at flight time
def get_faa_event_features(flight_time, events):
    """Return (max_duration_hours, active_count) for events active at flight_time."""
    active_durations = []
    for start, end in events:
        if start <= flight_time <= end:
            duration = (flight_time - start).total_seconds() / 3600
            active_durations.append(duration)
    if active_durations:
        return max(active_durations), len(active_durations)
    return 0.0, 0

# Compute flight timestamps
flight_times = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Hour'].astype(str) + ':00:00', errors='coerce')

durations = []
counts = []
for ft in flight_times:
    if pd.isna(ft):
        durations.append(0.0)
        counts.append(0)
    else:
        dur, cnt = get_faa_event_features(ft, all_events)
        durations.append(dur)
        counts.append(cnt)

df['faa_event_duration_hours'] = durations
df['faa_active_event_count'] = counts

print(f"\nFAA Event Depth:")
print(f"  faa_event_duration_hours: mean={df['faa_event_duration_hours'].mean():.2f}h, non-zero={df['faa_event_duration_hours'].gt(0).mean()*100:.1f}%")
print(f"  faa_active_event_count:   mean={df['faa_active_event_count'].mean():.2f}, non-zero={df['faa_active_event_count'].gt(0).mean()*100:.1f}%")

FAA events parsed: GDP=67, GSP=271, Total=338

FAA Event Depth:
  faa_event_duration_hours: mean=0.42h, non-zero=12.1%
  faa_active_event_count:   mean=0.17, non-zero=12.5%


## 8. LGA Cloud Ceiling & Pressure (V9.0)

In [15]:
# === V9.0: LGA Cloud Ceiling & Pressure Features ===
# lga_ceiling_low: OVC or BKN cloud cover = 1 (limits approach rate)
# lga_pressure_falling: pressure_desc in {Falling, Falling Rapidly} = 1 (storm precursor)

# These columns should already be merged from LGA weather in earlier cells
# If 'clds' column exists from weather merge:
if 'clds' in df.columns:
    df['lga_ceiling_low'] = df['clds'].isin(['OVC', 'BKN']).astype(int)
    print(f"lga_ceiling_low: {df['lga_ceiling_low'].mean()*100:.1f}% of flights have low ceiling")
else:
    # Fallback: merge from weather data
    print("WARNING: 'clds' column not found. Loading from weather files...")
    wx_files = sorted(DATA_RAW.glob('LGA_*_WeatherFull.csv'))
    wx_all = []
    for wf in wx_files:
        wx = pd.read_csv(wf)
        wx_all.append(wx)
    wx_df = pd.concat(wx_all, ignore_index=True)
    dt = pd.to_datetime(wx_df['valid_time_est'], errors='coerce')
    if dt.dt.tz is not None:
        dt = dt.dt.tz_localize(None)
    wx_df['Timestamp'] = dt.dt.floor('h')
    wx_df['Date'] = wx_df['Timestamp'].dt.date
    wx_df['Hour'] = wx_df['Timestamp'].dt.hour
    wx_hourly = wx_df.groupby(['Date', 'Hour']).last().reset_index()
    
    df['Date_date'] = pd.to_datetime(df['Date']).dt.date
    df = df.merge(wx_hourly[['Date', 'Hour', 'clds', 'pressure_desc']], 
                  left_on=['Date_date', 'Hour'], right_on=['Date', 'Hour'], 
                  how='left', suffixes=('', '_wx'))
    df['lga_ceiling_low'] = df['clds'].isin(['OVC', 'BKN']).astype(int)
    df.drop(columns=['Date_date', 'Date_wx'], inplace=True, errors='ignore')
    print(f"lga_ceiling_low: {df['lga_ceiling_low'].mean()*100:.1f}% of flights have low ceiling")

if 'pressure_desc' in df.columns:
    df['lga_pressure_falling'] = df['pressure_desc'].isin(['Falling', 'Falling Rapidly']).astype(int).fillna(0)
    print(f"lga_pressure_falling: {df['lga_pressure_falling'].mean()*100:.1f}% of flights (30% weather data has pressure_desc)")
else:
    df['lga_pressure_falling'] = 0
    print("WARNING: 'pressure_desc' not available, lga_pressure_falling set to 0")

lga_ceiling_low: 40.6% of flights have low ceiling
lga_pressure_falling: 9.7% of flights (30% weather data has pressure_desc)


## 9. Feature Summary

In [16]:
# List all features computed in 02 (V9.0)
new_features = (
    # Network features
    ['origin_impact_score', 'origin_historical_delay', 'origin_severe_delay_rate',
     'origin_risk_level', 'route_risk_score'] +
    # Gate/Terminal encoding (kept for backward compat, replaced by target encoding in 03)
    ['gate_encoded', 'terminal_encoded'] +
    # FAA Operational (expanded data)
    ['is_gdp_active', 'gdp_avg_delay', 'is_ground_stop',
     'is_dep_delay_active', 'faa_delay_severity', 'faa_delay_reason_raw'] +
    # Body Type
    ['body_type_narrow', 'body_type_commuter'] +
    # LGA Weather
    ['Precipitation', 'Wind_Gusts', 'Visibility', 'Is_Bad_Weather',
     'weather_severity_score', 'weather_category_encoded',
     'lga_storm_flag', 'lga_rain_flag', 'lga_fog_flag'] +
    # Origin Weather
    ['origin_precip', 'origin_wind_gust', 'origin_visibility', 'origin_bad_weather',
     'origin_pressure', 'origin_dewpoint', 'origin_temp',
     'origin_pressure_rapid_flag', 'origin_storm_flag', 'origin_cloud_cover'] +
    # Weather Impact Scores (V9.0)
    ['origin_wx_impact', 'lga_wx_impact']
)

print(f"Total features computed in 02: {len(new_features)}")
print("\nFeature availability:")
for i, f in enumerate(new_features, 1):
    if f in df.columns:
        null_pct = df[f].isna().sum() / len(df) * 100
        print(f"  {i:2}. {f}: {null_pct:.1f}% null")
    else:
        print(f"  {i:2}. {f}: NOT AVAILABLE")

Total features computed in 02: 36

Feature availability:
   1. origin_impact_score: 0.0% null
   2. origin_historical_delay: 0.0% null
   3. origin_severe_delay_rate: 0.0% null
   4. origin_risk_level: 0.0% null
   5. route_risk_score: 0.0% null
   6. gate_encoded: 0.0% null
   7. terminal_encoded: 0.0% null
   8. is_gdp_active: 0.0% null
   9. gdp_avg_delay: 0.0% null
  10. is_ground_stop: 0.0% null
  11. is_dep_delay_active: 0.0% null
  12. faa_delay_severity: 0.0% null
  13. faa_delay_reason_raw: 0.0% null
  14. body_type_narrow: 0.0% null
  15. body_type_commuter: 0.0% null
  16. Precipitation: 1.0% null
  17. Wind_Gusts: 1.0% null
  18. Visibility: 1.0% null
  19. Is_Bad_Weather: 1.0% null
  20. weather_severity_score: 0.0% null
  21. weather_category_encoded: 0.0% null
  22. lga_storm_flag: 1.0% null
  23. lga_rain_flag: 1.0% null
  24. lga_fog_flag: 1.0% null
  25. origin_precip: 69.0% null
  26. origin_wind_gust: 69.0% null
  27. origin_visibility: 69.0% null
  28. origin_bad_w

## 10. Save

In [17]:
# Save main feature-enriched data
output_path = DATA_PROCESSED / 'arrivals_with_features.parquet'
df.to_parquet(output_path, index=False)
print(f"Saved: {output_path} ({df.shape})")

# Save slimmed departure data for lag computation in 03
if df_departures is not None:
    dep_lag_cols = ['Scheduled_Departure_Datetime', 'Dep_Calculated_Delay', 'Dep_Taxi_Time',
                    'Dep_Date', 'Dep_Hour']
    dep_lag_cols = [c for c in dep_lag_cols if c in df_departures.columns]
    dep_lag_path = DATA_PROCESSED / 'departures_for_lag.parquet'
    df_departures[dep_lag_cols].to_parquet(dep_lag_path, index=False)
    print(f"Saved: {dep_lag_path} ({len(df_departures)} rows, {len(dep_lag_cols)} cols)")

# Save slimmed missed approaches data for lag computation in 03
if df_missed is not None:
    ma_lag_cols = ['MA_Datetime', 'MA_Date', 'MA_Hour']
    ma_lag_cols = [c for c in ma_lag_cols if c in df_missed.columns]
    ma_lag_path = DATA_PROCESSED / 'missed_approaches_for_lag.parquet'
    df_missed[ma_lag_cols].to_parquet(ma_lag_path, index=False)
    print(f"Saved: {ma_lag_path} ({len(df_missed)} rows, {len(ma_lag_cols)} cols)")

print("\n" + "=" * 60)
print("FEATURE ENGINEERING COMPLETE (V9.0 — Jan-Oct 2025 expanded data)")
print("=" * 60)
print(f"Main output: {output_path}")
print(f"Shape: {df.shape}")
print(f"Features computed in 02: {len(new_features)}")
print(f"\nV9.0 changes:")
print(f"  + origin_wx_impact, lga_wx_impact (Weather Impact Scores)")
print(f"  - origin_storm_flag, origin_cloud_cover, Month (removed in NB03+)")
print("Next: Notebook 03 adds lag features, target encoding, trains model")

Saved: ../../../data/processed/arrivals_with_features.parquet ((148165, 76))
Saved: ../../../data/processed/departures_for_lag.parquet (148061 rows, 5 cols)
Saved: ../../../data/processed/missed_approaches_for_lag.parquet (1051 rows, 3 cols)

FEATURE ENGINEERING COMPLETE (V9.0 — Jan-Oct 2025 expanded data)
Main output: ../../../data/processed/arrivals_with_features.parquet
Shape: (148165, 76)
Features computed in 02: 36

V9.0 changes:
  + origin_wx_impact, lga_wx_impact (Weather Impact Scores)
  - origin_storm_flag, origin_cloud_cover, Month (removed in NB03+)
Next: Notebook 03 adds lag features, target encoding, trains model
